# DDPM From Scratch — CelebA Faces (64×64 RGB)

A complete **Denoising Diffusion Probabilistic Model** for face generation on CelebA.

**Paper:** [Denoising Diffusion Probabilistic Models (Ho et al., 2020)](https://arxiv.org/abs/2006.11239)

### Architecture Highlights:
-  Large U-Net: 4 downsampling levels (64→32→16→8→4)
-  Channels: 128→128→256→256→512
-  Self-attention at 16×16 and 8×8
-  EMA with decay 0.9999
-  Gradient clipping, AdamW optimizer
-  Face generation, denoising visualization, interpolation

> **Note:** This notebook is designed for Google Colab T4 GPU. Training 100 epochs takes ~8-12 hours.
> Consider using Colab Pro for longer sessions, or reduce epochs to 50 for initial testing.

##  DDPM Theory Recap

### Why DDPM Works for Faces
Diffusion models learn the data distribution by reversing a gradual noising process.
For faces, this means the model learns:
1. **Global structure** first (face shape, background) at high noise levels
2. **Fine details** later (eyes, hair texture) at low noise levels

### Core Equations

**Forward (q):** Add noise progressively
$$x_t = \sqrt{\bar{\alpha}_t} \, x_0 + \sqrt{1 - \bar{\alpha}_t} \, \epsilon$$

**Reverse (p):** Denoise step by step
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{1 - \alpha_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sqrt{\beta_t} \, z$$

**Loss:** Simple noise prediction MSE
$$\mathcal{L} = \| \epsilon - \epsilon_\theta(\sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \epsilon, \, t) \|^2$$

In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import math
import copy
import os

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Hyperparameters
T = 1000
img_size = 64            # CelebA resized to 64x64
img_channels = 3
batch_size = 64          # Smaller batch for 64x64
epochs = 100
lr = 2e-4
base_channels = 128
ema_decay = 0.9999
grad_clip = 1.0

## 1. Dataset — CelebA (64×64)

CelebA contains ~200K celebrity face images. We center-crop to 140×140 and resize to 64×64.

> **Download:** CelebA will be downloaded automatically via `torchvision`.
> On Colab, this may take a few minutes. If the download fails, use the alternative Kaggle approach below.

In [ ]:
# Option 1: Download CelebA via torchvision (may hit Google Drive limits)
try:
    celeba_transform = transforms.Compose([
        transforms.CenterCrop(140),
        transforms.Resize(img_size),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x * 2 - 1),
    ])

    train_dataset = datasets.CelebA(
        root='./data', split='train', download=True, transform=celeba_transform
    )
    print(f'CelebA loaded: {len(train_dataset)} images')
except Exception as e:
    print(f'CelebA download failed: {e}')
    print('\nAlternative: Download from Kaggle:')
    print('1. !pip install kaggle')
    print('2. Upload kaggle.json to Colab')
    print('3. !kaggle datasets download -d jessicali9530/celeba-dataset')
    print('4. Use ImageFolder with the extracted path')

In [ ]:
# Alternative: Use ImageFolder if you have CelebA extracted locally
# Uncomment and modify the path as needed:
# import glob
# celeba_transform = transforms.Compose([
#     transforms.CenterCrop(140),
#     transforms.Resize(img_size),
#     transforms.ToTensor(),
#     transforms.Lambda(lambda x: x * 2 - 1),
# ])
# train_dataset = datasets.ImageFolder('./data/celeba/', transform=celeba_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)

In [ ]:
# Visualize training samples
batch = next(iter(train_loader))[0][:16]
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(16):
    ax = axes[i // 8, i % 8]
    img = ((batch[i].permute(1, 2, 0) + 1) / 2).clamp(0, 1).cpu()
    ax.imshow(img.numpy())
    ax.axis('off')
plt.suptitle('CelebA Training Samples (64×64)', fontsize=14)
plt.tight_layout(); plt.show()

## 2. Noise Schedule

In [ ]:
def linear_schedule(T, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, T)

def cosine_schedule(T, s=0.008):
    steps = torch.arange(T + 1, dtype=torch.float64)
    f_t = torch.cos(((steps / T) + s) / (1 + s) * (math.pi / 2)) ** 2
    alphas_bar = f_t / f_t[0]
    betas = 1 - (alphas_bar[1:] / alphas_bar[:-1])
    return torch.clamp(betas, 0.0001, 0.999).float()

betas = linear_schedule(T).to(device)
alphas = (1 - betas).to(device)
alphas_bars = torch.cumprod(alphas, dim=0).to(device)

# Plot
plt.figure(figsize=(8, 3))
plt.plot(alphas_bars.cpu().numpy())
plt.title('ᾱ_t Schedule'); plt.xlabel('t'); plt.ylabel('ᾱ_t')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
def forward_diffusion(x0, t):
    noise = torch.randn_like(x0)
    sqrt_ab = torch.sqrt(alphas_bars[t])[:, None, None, None]
    sqrt_1m_ab = torch.sqrt(1 - alphas_bars[t])[:, None, None, None]
    return sqrt_ab * x0 + sqrt_1m_ab * noise, noise

# Visualize forward diffusion on a face
sample_face = next(iter(train_loader))[0][0:1].to(device)
ts = [0, 50, 100, 250, 500, 750, 999]
fig, axes = plt.subplots(1, len(ts), figsize=(16, 2.5))
for i, tv in enumerate(ts):
    noisy, _ = forward_diffusion(sample_face, torch.tensor([tv]).to(device))
    img = ((noisy[0].cpu().permute(1, 2, 0) + 1) / 2).clamp(0, 1)
    axes[i].imshow(img.numpy()); axes[i].set_title(f't={tv}'); axes[i].axis('off')
plt.suptitle('Forward Diffusion on a Face', fontsize=14)
plt.tight_layout(); plt.show()

## 3. U-Net Architecture (64×64)

For 64×64 images we need a deeper U-Net:
- **4 encoder levels**: 64→32→16→8→4 (bottleneck)
- **Channels**: 128→128→256→256→512
- **Self-attention** at 16×16 and 8×8
- **2 ResBlocks per level** for more capacity

In [ ]:
class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t[:, None] * freqs[None, :]
        return torch.cat((torch.sin(args), torch.cos(args)), dim=-1)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Linear(time_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_mlp(t)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, channels, n_heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.n_heads = n_heads
        self.head_dim = channels // n_heads
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, self.n_heads, self.head_dim, H * W)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        attn = torch.softmax((q.transpose(-1, -2) @ k) / math.sqrt(self.head_dim), dim=-1)
        h = (attn @ v.transpose(-1, -2)).transpose(-1, -2).reshape(B, C, H, W)
        return x + self.proj(h)

In [ ]:
class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)
    def forward(self, x): return self.conv(x)

class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)
    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2, mode='nearest'))

In [ ]:
class UNet64(nn.Module):
    """
    U-Net for 64×64 CelebA DDPM.
    Encoder: 64→32→16→8→4
    Channels: 128→128→256→256→512
    Attention at 16×16 and 8×8
    """
    def __init__(self, in_ch=3, base_ch=128):
        super().__init__()
        ch = base_ch
        td = ch * 4  # time dim

        self.time_mlp = nn.Sequential(
            TimeEmbedding(ch), nn.Linear(ch, td), nn.SiLU(), nn.Linear(td, td))

        self.conv_in = nn.Conv2d(in_ch, ch, 3, padding=1)

        # Enc Level 1: 64→32, ch→ch (128)
        self.e1r1 = ResBlock(ch, ch, td)
        self.e1r2 = ResBlock(ch, ch, td)
        self.d1 = Downsample(ch)

        # Enc Level 2: 32→16, ch→ch*2 (256)
        self.e2r1 = ResBlock(ch, ch * 2, td)
        self.e2r2 = ResBlock(ch * 2, ch * 2, td)
        self.d2 = Downsample(ch * 2)

        # Enc Level 3: 16→8, ch*2→ch*2 (256), with attention
        self.e3r1 = ResBlock(ch * 2, ch * 2, td)
        self.e3a = SelfAttention(ch * 2)
        self.e3r2 = ResBlock(ch * 2, ch * 2, td)
        self.d3 = Downsample(ch * 2)

        # Enc Level 4: 8→4, ch*2→ch*2 (256), with attention
        self.e4r1 = ResBlock(ch * 2, ch * 2, td)
        self.e4a = SelfAttention(ch * 2)
        self.e4r2 = ResBlock(ch * 2, ch * 2, td)
        self.d4 = Downsample(ch * 2)

        # Bottleneck: 4×4, ch*4 (512)
        self.mr1 = ResBlock(ch * 2, ch * 4, td)
        self.ma = SelfAttention(ch * 4)
        self.mr2 = ResBlock(ch * 4, ch * 4, td)

        # Dec Level 4: 4→8
        self.u4 = Upsample(ch * 4)
        self.d4r1 = ResBlock(ch * 4 + ch * 2, ch * 2, td)
        self.d4a = SelfAttention(ch * 2)
        self.d4r2 = ResBlock(ch * 2, ch * 2, td)

        # Dec Level 3: 8→16
        self.u3 = Upsample(ch * 2)
        self.d3r1 = ResBlock(ch * 2 + ch * 2, ch * 2, td)
        self.d3a = SelfAttention(ch * 2)
        self.d3r2 = ResBlock(ch * 2, ch * 2, td)

        # Dec Level 2: 16→32
        self.u2 = Upsample(ch * 2)
        self.d2r1 = ResBlock(ch * 2 + ch * 2, ch * 2, td)
        self.d2r2 = ResBlock(ch * 2, ch, td)

        # Dec Level 1: 32→64
        self.u1 = Upsample(ch)
        self.d1r1 = ResBlock(ch + ch, ch, td)
        self.d1r2 = ResBlock(ch, ch, td)

        self.norm_out = nn.GroupNorm(8, ch)
        self.conv_out = nn.Conv2d(ch, in_ch, 3, padding=1)

    def forward(self, x, t):
        t = self.time_mlp(t.float())
        x = self.conv_in(x)

        # Encoder
        s1 = self.e1r2(self.e1r1(x, t), t); h = self.d1(s1)
        s2 = self.e2r2(self.e2r1(h, t), t); h = self.d2(s2)
        s3 = self.e3r2(self.e3a(self.e3r1(h, t)), t); h = self.d3(s3)
        s4 = self.e4r2(self.e4a(self.e4r1(h, t)), t); h = self.d4(s4)

        # Bottleneck
        h = self.mr2(self.ma(self.mr1(h, t)), t)

        # Decoder
        h = self.u4(h); h = torch.cat([h, s4], 1)
        h = self.d4r2(self.d4a(self.d4r1(h, t)), t)

        h = self.u3(h); h = torch.cat([h, s3], 1)
        h = self.d3r2(self.d3a(self.d3r1(h, t)), t)

        h = self.u2(h); h = torch.cat([h, s2], 1)
        h = self.d2r2(self.d2r1(h, t), t)

        h = self.u1(h); h = torch.cat([h, s1], 1)
        h = self.d1r2(self.d1r1(h, t), t)

        return self.conv_out(F.silu(self.norm_out(h)))

## 4. EMA & Model Setup

In [ ]:
class EMA:
    def __init__(self, model, decay=0.9999):
        self.decay = decay
        self.shadow = copy.deepcopy(model)
        self.shadow.eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for s, m in zip(self.shadow.parameters(), model.parameters()):
            s.data.mul_(self.decay).add_(m.data, alpha=1 - self.decay)

    def __call__(self, *a, **kw): return self.shadow(*a, **kw)

In [ ]:
model = UNet64(in_ch=img_channels, base_ch=base_channels).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
ema = EMA(model, decay=ema_decay)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 5. Training

Training CelebA 64×64 takes ~8-12 hours on a T4 for 100 epochs.
Loss should converge to ~0.02-0.03 range.

In [ ]:
losses = []

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for x, _ in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}'):
        x = x.to(device)
        t = torch.randint(0, T, (x.shape[0],)).to(device)

        optimizer.zero_grad()
        xt, noise = forward_diffusion(x, t)
        pred = model(xt, t)
        loss = F.mse_loss(pred, noise)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        ema.update(model)

        total_loss += loss.item()

    avg = total_loss / len(train_loader)
    losses.append(avg)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch+1}/{epochs} | Loss: {avg:.6f}')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses, linewidth=1.5, color='#e74c3c')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('DDPM Training Loss — CelebA 64×64')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 6. Generate Faces

In [ ]:
@torch.no_grad()
def sample_ddpm(model_fn, n=16, return_steps=False):
    x = torch.randn(n, img_channels, img_size, img_size).to(device)
    steps = []

    for t_val in tqdm(reversed(range(T)), total=T, desc='Sampling'):
        tb = torch.full((n,), t_val, device=device)
        eps = model_fn(x, tb)
        a = alphas[t_val]; ab = alphas_bars[t_val]; b = betas[t_val]
        z = torch.randn_like(x) if t_val > 0 else torch.zeros_like(x)
        x = (1 / torch.sqrt(a)) * (x - (1 - a) / torch.sqrt(1 - ab) * eps) + torch.sqrt(b) * z

        if return_steps and t_val % (T // 10) == 0:
            steps.append(x.clone())

    return (x, steps) if return_steps else x

In [ ]:
# Generate faces with EMA model
ema.shadow.eval()
faces = sample_ddpm(ema, n=16)
faces = ((faces + 1) / 2).clamp(0, 1)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for i in range(16):
    ax = axes[i // 4, i % 4]
    ax.imshow(faces[i].cpu().permute(1, 2, 0).numpy())
    ax.axis('off')
plt.suptitle('Generated Faces (EMA Model)', fontsize=16)
plt.tight_layout(); plt.show()

## 7. Denoising Process Visualization

In [ ]:
ema.shadow.eval()
_, denoise_steps = sample_ddpm(ema, n=4, return_steps=True)

n_s = len(denoise_steps)
fig, axes = plt.subplots(4, n_s, figsize=(2 * n_s, 8))
for row in range(4):
    for col, si in enumerate(denoise_steps):
        img = ((si[row].cpu().permute(1, 2, 0) + 1) / 2).clamp(0, 1)
        axes[row, col].imshow(img.numpy())
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(f't≈{T - col * (T // 10)}', fontsize=9)
plt.suptitle('Face Generation: Noise → Face', fontsize=14)
plt.tight_layout(); plt.show()

## 8. Large Sample Grid

In [ ]:
ema.shadow.eval()
big = sample_ddpm(ema, n=64)
big = ((big + 1) / 2).clamp(0, 1)

fig, axes = plt.subplots(8, 8, figsize=(14, 14))
for i in range(64):
    ax = axes[i // 8, i % 8]
    ax.imshow(big[i].cpu().permute(1, 2, 0).numpy())
    ax.axis('off')
plt.suptitle('64 Generated Faces — CelebA DDPM', fontsize=16)
plt.tight_layout(); plt.show()

## 9. Latent Space Interpolation

Interpolate between two noise vectors to see smooth face transitions.

In [ ]:
@torch.no_grad()
def interpolate(model_fn, n_interp=8):
    z1 = torch.randn(1, img_channels, img_size, img_size).to(device)
    z2 = torch.randn(1, img_channels, img_size, img_size).to(device)

    alphas_interp = torch.linspace(0, 1, n_interp).to(device)
    results = []

    for a in alphas_interp:
        z = (1 - a) * z1 + a * z2  # Spherical would be better, but linear works visually
        x = z.clone()
        for t_val in reversed(range(T)):
            tb = torch.full((1,), t_val, device=device)
            eps = model_fn(x, tb)
            al = alphas[t_val]; ab = alphas_bars[t_val]; b = betas[t_val]
            noise = torch.randn_like(x) if t_val > 0 else torch.zeros_like(x)
            x = (1 / torch.sqrt(al)) * (x - (1 - al) / torch.sqrt(1 - ab) * eps) + torch.sqrt(b) * noise
        results.append(((x[0] + 1) / 2).clamp(0, 1).cpu())

    fig, axes = plt.subplots(1, n_interp, figsize=(2 * n_interp, 2.5))
    for i, img in enumerate(results):
        axes[i].imshow(img.permute(1, 2, 0).numpy())
        axes[i].axis('off')
        axes[i].set_title(f'α={alphas_interp[i]:.2f}', fontsize=8)
    plt.suptitle('Latent Space Interpolation', fontsize=14)
    plt.tight_layout(); plt.show()

ema.shadow.eval()
interpolate(ema)